# Bài 48: Bộ tự mã hóa (Autoencoders)

Mọi mạng cho đến giờ đều học từ các mẫu có nhãn: một ảnh và lớp, hộp hoặc mặt nạ đúng của nó. **Bộ tự mã hóa (autoencoder)** không cần bất kỳ thứ nào trong số đó — chúng là một ví dụ nền tảng của **học không giám sát**, nơi tín hiệu huấn luyện duy nhất đến từ chính dữ liệu, không bao giờ từ một nhãn do con người cung cấp. Huấn luyện một mạng tái tạo chính đầu vào của nó, qua một nút thắt cổ chai hẹp có chủ đích, và nút thắt cổ chai đó buộc phải khám phá ra một biểu diễn nén, có ý nghĩa của dữ liệu — không phải vì ai đó đã gán nhãn gì, mà vì việc tái tạo qua một đường ống đủ hẹp chỉ có thể giải được bằng cách vứt bỏ việc một hình dạng trông như thế nào đến từng điểm ảnh và giữ lại việc một hình dạng *là* gì. Bài này xây dựng một bộ tự mã hóa từ đầu (<a href="../references.html#hinton-2006-autoencoder">Hinton & Salakhutdinov, 2006</a>), so sánh nó với phương án tuyến tính cổ điển (PCA), và cho thấy nút thắt cổ chai thực sự mang lại gì: khử nhiễu, và một không gian tiềm ẩn (latent space) có thể nội suy một cách mượt mà.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Một tập dữ liệu hình dạng, và một bộ mã hóa-giải mã

Dùng lại các hình dấu cộng/hình tròn từ các bài trước. **Bộ mã hóa** nén một ảnh 16x16 xuống một vector thắt cổ chai nhỏ; **bộ giải mã** mở rộng vector đó trở lại thành một bản tái tạo 16x16 đầy đủ. Huấn luyện cực tiểu hóa sai số tái tạo — đầu vào cũng là mục tiêu, nên không có nhãn nào được dùng ở bất kỳ đâu trong bài này.

In [ ]:
def make_image(shape_type, cx, cy, size=16):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        img[cy-1:cy+2, cx-3:cx+4] = 1.0
        img[cy-3:cy+4, cx-1:cx+2] = 1.0
    else:
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx-cx)**2 + (yy-cy)**2) <= 9] = 1.0
    return img

def make_dataset(rng_local, n, position_range=(4, 12)):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        imgs.append(make_image(shape_type, cx, cy))
        labels.append(0 if shape_type == 'plus' else 1)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.int64)

rng = np.random.default_rng(6)
X_train, y_train = make_dataset(rng, 400)
X_test, y_test = make_dataset(rng, 150)

class Encoder(nn.Module):
    def __init__(self, bottleneck=8):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16 -> 8
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 8 -> 4
        )
        self.fc = nn.Linear(16 * 4 * 4, bottleneck)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))

class Decoder(nn.Module):
    def __init__(self, bottleneck=8):
        super().__init__()
        self.fc = nn.Linear(bottleneck, 16 * 4 * 4)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.conv1 = nn.Sequential(nn.Conv2d(16, 8, 3, padding=1), nn.ReLU())  # 4 -> 8
        self.conv2 = nn.Conv2d(8, 1, 3, padding=1)                             # 8 -> 16

    def forward(self, z):
        h = self.fc(z).view(-1, 16, 4, 4)
        h = self.conv1(self.up(h))
        return torch.sigmoid(self.conv2(self.up(h)))

class Autoencoder(nn.Module):
    def __init__(self, bottleneck=8):
        super().__init__()
        self.encoder = Encoder(bottleneck)
        self.decoder = Decoder(bottleneck)

    def forward(self, x):
        return self.decoder(self.encoder(x))

def train_autoencoder(bottleneck, epochs=300, lr=0.01, seed=0):
    torch.manual_seed(seed)
    model = Autoencoder(bottleneck)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt = torch.tensor(X_train).unsqueeze(1)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.mse_loss(model(Xt), Xt)
        loss.backward()
        opt.step()
    return model

model = train_autoencoder(bottleneck=8)
Xte = torch.tensor(X_test).unsqueeze(1)
with torch.no_grad():
    recon_te = model(Xte)
    test_mse = F.mse_loss(recon_te, Xte).item()
print(f'test reconstruction MSE (bottleneck=8): {test_mse:.4f}')

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for i in range(6):
    axes[0, i].imshow(X_test[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(recon_te[i, 0].numpy(), cmap='gray'); axes[1, i].axis('off')
axes[0, 0].set_title('original', fontsize=9, loc='left')
axes[1, 0].set_title('reconstructed', fontsize=9, loc='left')
plt.show()

## Nút thắt cổ chai quan trọng đến đâu?

Một nút thắt cổ chai 8 số cho một ảnh 256 điểm ảnh là một phép nén 32 lần. Hãy quét độ rộng nút thắt cổ chai và quan sát chất lượng tái tạo đánh đổi trực tiếp với nó.

In [ ]:
bottlenecks = [1, 2, 4, 8, 16, 32]
ae_mses = []
for b in bottlenecks:
    m = train_autoencoder(bottleneck=b, seed=0)
    with torch.no_grad():
        mse = F.mse_loss(m(Xte), Xte).item()
    ae_mses.append(mse)
    print(f'bottleneck={b:>3}: test MSE = {mse:.4f}')

plt.figure(figsize=(5, 3.5))
plt.plot(bottlenecks, ae_mses, '-o')
plt.xscale('log', base=2)
plt.xlabel('bottleneck size'); plt.ylabel('test reconstruction MSE')
plt.title('Reconstruction quality vs. bottleneck width')
plt.show()

## Nén phi tuyến và tuyến tính: bộ tự mã hóa so với PCA

PCA là phiên bản cổ điển của cùng ý tưởng này: chiếu lên k hướng có phương sai lớn nhất, rồi chiếu ngược lại — nhưng chỉ luôn qua một ánh xạ *tuyến tính*. Hãy so sánh chất lượng tái tạo ở cùng kích thước nút thắt cổ chai.

In [ ]:
def pca_reconstruct(X_train_flat, X_test_flat, k):
    mean = X_train_flat.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(X_train_flat - mean, full_matrices=False)
    components = Vt[:k]  # (k, 256)
    projected = (X_test_flat - mean) @ components.T
    return projected @ components + mean

X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

pca_mses = []
for k in bottlenecks:
    recon_pca = pca_reconstruct(X_train_flat, X_test_flat, k)
    pca_mses.append(np.mean((recon_pca - X_test_flat) ** 2))

print(f'{"bottleneck":>10} {"autoencoder MSE":>16} {"PCA MSE":>10}')
for b, ae_mse, pca_mse in zip(bottlenecks, ae_mses, pca_mses):
    print(f'{b:>10} {ae_mse:>16.4f} {pca_mse:>10.4f}')

Ở mọi nút thắt cổ chai nhỏ (1, 2, 4, 8), bộ tự mã hóa phi tuyến tái tạo tốt hơn thấy rõ so với PCA ở cùng số chiều — một phép chiếu tuyến tính thực sự không thể nắm bắt "danh tính hình dạng cộng một vị trí 2D" hiệu quả bằng một mạng có thể uốn phép nén của nó quanh cấu trúc cong thực sự của dữ liệu. Nhưng hãy nhìn điều gì xảy ra ở nút thắt cổ chai 32: sai số của bộ tự mã hóa đã **chững lại** (nó ngừng cải thiện từ nút thắt cổ chai 4), trong khi sai số của PCA tiếp tục giảm khi nó được cho thêm nhiều hướng tuyến tính để dùng — và ở 32 chiều PCA thực sự đã vượt qua nó. Đây không phải là khiếm khuyết của ý tưởng phi tuyến; đó là lời nhắc rằng "phi tuyến thắng tuyến tính" chỉ đúng ở nơi tính phi tuyến là ràng buộc then chốt. Khi nút thắt cổ chai đủ rộng để năng lực không còn là nút thắt cổ chai nữa, sai số còn lại của bộ tự mã hóa đến từ *huấn luyện* — tối ưu một mạng phi tuyến thực sự là bài toán khó hơn nghiệm chính xác, dạng đóng của PCA — chứ không phải từ một mức trần cơ bản nào về những gì một mạng có thể biểu diễn.

## Khử nhiễu: tái tạo tín hiệu sạch từ một đầu vào bị làm hỏng

Một bộ tự mã hóa khử nhiễu (denoising autoencoder) được huấn luyện với một điểm xoắn: đưa vào một đầu vào *bị làm hỏng*, nhưng vẫn yêu cầu nó tái tạo bản gốc *sạch*. Làm tốt điều này buộc nút thắt cổ chai phải mã hóa tín hiệu, không phải nhiễu, vì bản thân nhiễu không có gì dự đoán được để bảo toàn.

In [ ]:
def add_noise(imgs, rng_local, level=0.3):
    noisy = imgs + rng_local.normal(0, level, imgs.shape)
    return np.clip(noisy, 0, 1).astype(np.float32)

noise_rng = np.random.default_rng(9)
X_train_noisy = add_noise(X_train, noise_rng)
X_test_noisy = add_noise(X_test, noise_rng)

def train_denoising_autoencoder(bottleneck=8, epochs=300, lr=0.01, seed=0):
    torch.manual_seed(seed)
    model = Autoencoder(bottleneck)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt_noisy = torch.tensor(X_train_noisy).unsqueeze(1)
    Xt_clean = torch.tensor(X_train).unsqueeze(1)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.mse_loss(model(Xt_noisy), Xt_clean)
        loss.backward()
        opt.step()
    return model

denoiser = train_denoising_autoencoder()
Xte_noisy = torch.tensor(X_test_noisy).unsqueeze(1)
with torch.no_grad():
    denoised = denoiser(Xte_noisy)

noisy_vs_clean_mse = np.mean((X_test_noisy - X_test) ** 2)
denoised_vs_clean_mse = F.mse_loss(denoised, Xte).item()
print(f'noisy input vs. clean target MSE:     {noisy_vs_clean_mse:.4f}  (doing nothing)')
print(f'denoiser output vs. clean target MSE: {denoised_vs_clean_mse:.4f}')

fig, axes = plt.subplots(3, 6, figsize=(11, 5.5))
for i in range(6):
    axes[0, i].imshow(X_test[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(X_test_noisy[i], cmap='gray'); axes[1, i].axis('off')
    axes[2, i].imshow(denoised[i, 0].numpy(), cmap='gray'); axes[2, i].axis('off')
for r, name in enumerate(['clean', 'noisy input', 'denoised']):
    axes[r, 0].set_title(name, fontsize=9, loc='left')
plt.show()

## Một không gian tiềm ẩn có cấu trúc: nội suy

Nếu nút thắt cổ chai thực sự mã hóa danh tính hình dạng và vị trí thay vì ghi nhớ từng điểm ảnh riêng lẻ, việc giải mã một điểm *giữa* hai ảnh đã mã hóa sẽ tạo ra thứ gì đó ở giữa, không phải nhiễu.

In [ ]:
idx_a = 0
idx_b = next(i for i in range(len(X_test)) if y_test[i] != y_test[idx_a])

with torch.no_grad():
    z_a = model.encoder(Xte[idx_a:idx_a+1])
    z_b = model.encoder(Xte[idx_b:idx_b+1])
    alphas = np.linspace(0, 1, 6)
    interp_imgs = [model.decoder((1 - a) * z_a + a * z_b)[0, 0].numpy() for a in alphas]

shape_names = ['plus', 'circle']
fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, im, a in zip(axes, interp_imgs, alphas):
    ax.imshow(im, cmap='gray'); ax.axis('off')
    ax.set_title(f'{a:.1f}', fontsize=8)
plt.suptitle(f'Interpolating in latent space: {shape_names[y_test[idx_a]]} -> {shape_names[y_test[idx_b]]}')
plt.show()

## Tiếp theo là gì

Không phần nào trong bài này dùng đến một nhãn nào — mọi kết quả đều đến chỉ từ mục tiêu tái tạo. Mẫu hình đó, một mạng bị buộc phải nén rồi xây dựng lại chính đầu vào của nó, là tổ tiên của một lượng đáng ngạc nhiên những gì phần còn lại của phần này làm với dữ liệu không nhãn: học tương phản (contrastive learning) ở Bài 49 và tự chưng cất (self-distillation) ở Bài 50 đều thay "tái tạo các điểm ảnh" bằng các mục tiêu tự giám sát khác, nhưng giữ cùng cấu trúc "không nhãn, tự tạo ra tín hiệu huấn luyện từ chính dữ liệu". Bài 51 xây dựng một bộ tự mã hóa **có che (masked)** — cùng hình dạng mã hóa-giải mã như bài này, nhưng với *các mảng bị che ngẫu nhiên* làm nút thắt cổ chai thông tin thay vì một vector hẹp. Và mục tiêu khử nhiễu được xây dựng ở đây, áp dụng mỗi lần một bước nhiễu nhỏ và nối chuỗi với nhau, chính xác là cơ chế mà các mô hình khuếch tán (diffusion model) ở Bài 60 mở rộng quy mô để sinh ra các ảnh hoàn toàn mới từ nhiễu thuần túy.

### Bài tập

1. Thử `bottleneck=64` và `bottleneck=128` trong phép quét ở trên. MSE tái tạo có tiếp tục cải thiện mãi, hay chững lại — và một sự chững lại gợi ý gì về số bậc tự do mà tập dữ liệu này thực sự có (danh tính hình dạng cộng một vị trí 2D)?
2. Bộ tự mã hóa khử nhiễu ở trên được huấn luyện và đánh giá ở `level=0.3`. Hãy huấn luyện lại và đánh giá lại ở `level=0.6`. Bộ khử nhiễu có còn khôi phục được thứ gì nhận ra được không, hay phép tái tạo sụp đổ thành một vệt mờ mơ hồ?
3. Huấn luyện lại một bộ tự mã hóa với `bottleneck=2`, mã hóa cả 150 ảnh kiểm tra, và vẽ biểu đồ phân tán các mã 2D thu được tô màu theo `y_test`. Ảnh dấu cộng và hình tròn có tách thành hai cụm thấy rõ không, dù mạng chưa bao giờ thấy một nhãn nào trong khi huấn luyện?